# 24.8 设计 RAG 系统 / Design a RAG System (企业知识库问答 / Enterprise Knowledge Base QA)

**中文**:大语言模型(LLM)很强,但有两个致命短板:①**不知道你的私有/最新数据**(它只学过公开的、截止某个日期的训练数据,不知道你公司的内部文档、今天的新闻);②**会一本正经地胡编(幻觉)**。**RAG(Retrieval-Augmented Generation,检索增强生成)** 是解决这两点的主流方案,也是当下最热门的 LLM 应用架构:*在让 LLM 回答问题之前,先从你的知识库里检索相关文档,把它们塞进 LLM 的上下文,让 LLM"看着资料回答"*。这样它既能用你的私有/最新数据,又因为有依据而大幅减少幻觉。设计"企业知识库问答"是当下最高频的 LLM 系统设计题。它有一个决定性的洞察:**RAG 系统的答案质量,几乎完全由检索质量决定——检索到垃圾,再强的 LLM 也只能基于垃圾回答(或幻觉)。** 本节从零实现一个 mini RAG,亲眼看到检索如何"锚定"答案、以及检索失败时的风险,再讲清 RAG 系统设计的完整框架。
**English**: Large language models (LLMs) are powerful but have two fatal weaknesses: ① **they don't know your private/latest data** (trained only on public data up to a cutoff date, unaware of your internal docs or today's news); ② **they confidently make things up (hallucinate)**. **RAG (Retrieval-Augmented Generation)** is the mainstream solution to both and the hottest LLM application architecture today: *before letting the LLM answer, retrieve relevant documents from your knowledge base and stuff them into the LLM's context, so it "answers while looking at the source material"*. This lets it use your private/latest data and, being grounded, greatly reduces hallucination. Designing "enterprise knowledge-base QA" is currently the most frequent LLM system-design question. It has a decisive insight: **a RAG system's answer quality is almost entirely determined by retrieval quality — retrieve garbage and even the best LLM can only answer based on garbage (or hallucinate).** This section implements a mini RAG from scratch, seeing how retrieval "anchors" the answer and the risk when retrieval fails, then clarifies the complete RAG-system-design framework.

---

**中文**:**RAG 的流水线(要能画出来)**:
**English**: **The RAG pipeline (be able to draw it)**:
1. **中文**:**离线:构建知识库**。把文档**切块(chunking)**——分成合适大小的片段(太大→检索不精准、塞不进上下文;太小→丢失语境)。每块用 **embedding 模型编码成向量**,存入**向量数据库(vector DB)+ ANN 索引**。
   **Offline: build the knowledge base**. **Chunk** documents into appropriately-sized pieces (too big → imprecise retrieval, won't fit context; too small → loses context). Encode each chunk into a vector with an **embedding model**, storing them in a **vector database + ANN index**.
2. **中文**:**在线:检索(retrieve)**。用户提问 → 把问题也编码成向量 → 在向量库里找**最相似的 top-k 块**(常混合语义 + 关键词检索,接 24.2)。
   **Online: retrieve**. User asks → encode the question into a vector → find the **top-k most similar chunks** in the vector store (often hybrid semantic + keyword, per 24.2).
3. **中文**:**重排(rerank)**:用一个更强的模型(cross-encoder)对召回的 top-k 重新精排,选出最相关的几块(检索召回要广,重排要准)。
   **Rerank**: use a stronger model (cross-encoder) to re-rank the retrieved top-k, selecting the most relevant few (retrieval recall broad, rerank precise).
4. **中文**:**生成(generate)**:把"用户问题 + 检索到的相关块"一起塞进 LLM 的提示词,让 LLM **基于这些资料**生成答案(并最好**引用来源**)。
   **Generate**: put "user question + retrieved relevant chunks" together into the LLM's prompt, letting the LLM generate an answer **based on this material** (ideally **citing sources**).

**中文**:**核心洞察:检索是瓶颈,不是生成**。人们常以为 RAG 的难点在"用什么 LLM",其实**答案质量的上限由检索决定**:①如果没检索到相关内容,LLM 要么答不出、要么幻觉;②如果检索到错误内容,LLM 会基于错误资料给出错误答案(而且听起来很自信)。所以 RAG 系统 80% 的工程功夫在**把检索做好**:切块策略、embedding 质量、混合检索、重排、以及**识别"知识库里没有答案"时诚实地说"我不知道"**(而非硬编造)。
**English**: **Core insight: retrieval is the bottleneck, not generation**. People often think RAG's difficulty is "which LLM," but **the ceiling on answer quality is set by retrieval**: ① if relevant content isn't retrieved, the LLM either can't answer or hallucinates; ② if wrong content is retrieved, the LLM gives a wrong answer based on wrong material (and sounds confident). So 80% of RAG engineering is **getting retrieval right**: chunking strategy, embedding quality, hybrid retrieval, reranking, and **honestly saying "I don't know" when the knowledge base has no answer** (rather than making it up).

> 💡 **面试速查 / Interview cheat-sheet（★★★ LLM 系统设计, 当下最热）**
> **中文**:**RAG=检索增强生成**:回答前从知识库检索相关文档塞进 LLM 上下文, 解决 LLM 不知道私有/最新数据 + 减少幻觉。**流水线**:①**离线**:文档**切块(chunking)**→embedding 编码→存**向量库+ANN 索引**;②**检索**:问题编码→top-k 相似块(混合语义+关键词, 24.2);③**重排**:cross-encoder 精排;④**生成**:问题+检索块→LLM 生成(引用来源)。**核心洞察=检索决定答案质量**(检索到垃圾→垃圾答案/幻觉), 80% 功夫在检索。**关键决策**:切块大小/重叠(太大不精准/太小丢语境)、embedding 模型、**混合检索**(语义+BM25)、重排、top-k、上下文窗口预算。**关键难题**:①**幻觉**——即使有 RAG 也可能编造→引用来源+"没有依据就说不知道"+忠实度评估;②**检索失败**(相关内容没召回/切块切碎了答案)→评估检索召回;③**评估难**——分检索指标(recall@k)和生成指标(忠实度faithfulness/答案相关性, 用 RAGAS/LLM-as-judge);④更新知识库(增量索引)、权限(用户只能看有权限的文档)、多跳问题、表格/图像。**RAG vs 微调**:RAG 加知识(动态、可溯源、便宜), 微调改行为/风格; 常结合。**Agentic RAG**:LLM 自主决定检索什么、多轮检索。面试金句:*"RAG 在 LLM 回答前检索知识库相关块塞进上下文, 解决私有/最新数据和幻觉; 流水线是切块→embedding→向量库检索→重排→生成引用来源; 核心洞察是检索质量决定答案质量(80%功夫在检索:切块/混合检索/重排), 检索不到要诚实说不知道防幻觉; 评估分检索召回和生成忠实度; 加知识用 RAG、改行为用微调。"*
> **English**: **RAG = Retrieval-Augmented Generation**: before answering, retrieve relevant documents from a knowledge base into the LLM's context, solving the LLM not knowing private/latest data + reducing hallucination. **Pipeline**: ① **offline**: **chunk** documents → embedding encode → store in a **vector DB + ANN index**; ② **retrieve**: encode question → top-k similar chunks (hybrid semantic + keyword, 24.2); ③ **rerank**: cross-encoder refinement; ④ **generate**: question + retrieved chunks → LLM generates (cite sources). **Core insight = retrieval determines answer quality** (retrieve garbage → garbage answer/hallucination), 80% of effort is in retrieval. **Key decisions**: chunk size/overlap (too big imprecise / too small loses context), embedding model, **hybrid retrieval** (semantic + BM25), reranking, top-k, context-window budget. **Key challenges**: ① **hallucination** — even with RAG it may fabricate → cite sources + "say I don't know if no basis" + faithfulness evaluation; ② **retrieval failure** (relevant content not recalled / chunking split the answer) → evaluate retrieval recall; ③ **hard evaluation** — separate retrieval metrics (recall@k) and generation metrics (faithfulness/answer relevance, via RAGAS/LLM-as-judge); ④ knowledge-base updates (incremental indexing), permissions (users see only authorized docs), multi-hop questions, tables/images. **RAG vs fine-tuning**: RAG adds knowledge (dynamic, traceable, cheap), fine-tuning changes behavior/style; often combined. **Agentic RAG**: the LLM decides what to retrieve, multi-round retrieval. Interview line: *"RAG retrieves relevant knowledge-base chunks into the LLM's context before answering, solving private/latest data and hallucination; the pipeline is chunk → embedding → vector-DB retrieval → rerank → generate with source citations; the core insight is retrieval quality determines answer quality (80% of effort in retrieval: chunking/hybrid retrieval/reranking), and it should honestly say 'I don't know' when nothing is retrieved to prevent hallucination; evaluate retrieval recall and generation faithfulness separately; add knowledge with RAG, change behavior with fine-tuning."*


In [ ]:

# ============================================================
# 从零实现 mini RAG:检索如何锚定答案 / mini RAG from scratch: how retrieval grounds the answer
# 中文:一个小知识库(企业 FAQ)。把每条切块→用 TF-IDF 编码成向量(生产用语义 embedding)。
#      用户提问→检索最相关的块→这些块作为 LLM 的上下文。演示: 有答案时检索锚定, 没答案时检索分数低(应说'不知道')。
# English: a small knowledge base (enterprise FAQ). Each chunk encoded via TF-IDF (production uses semantic embeddings).
#      User asks → retrieve most relevant chunks → these ground the LLM. Show: retrieval anchors when answer exists; scores low when it doesn't (should say "I don't know").
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
knowledge_base=[
 "The company refund policy allows returns within 30 days of purchase with a receipt.",
 "Refunds are processed within 5-7 business days to the original payment method.",
 "Premium subscription costs $99 per year and includes priority support.",
 "To reset your password, click 'Forgot Password' on the login page and check your email.",
 "Our office is at 123 Main Street, open 9am to 5pm on weekdays.",
 "The annual company retreat is held every summer in the mountains.",
]
vectorizer=TfidfVectorizer(stop_words="english").fit(knowledge_base)   # embedding 模型(玩具版)/ toy embedding model
kb_vectors=vectorizer.transform(knowledge_base)                        # 每块的向量(存入向量库)/ chunk vectors (in vector store)
def retrieve(query, k=3):                                              # 检索最相似的 top-k 块 / retrieve top-k similar chunks
    qv=vectorizer.transform([query]); sims=cosine_similarity(qv, kb_vectors)[0]
    return [(knowledge_base[i], round(float(sims[i]),3)) for i in np.argsort(-sims)[:k]]

# 情形①:知识库里有答案 → 检索锚定, LLM 有依据地回答 / case ①: answer exists → retrieval anchors, LLM answers grounded
q1="How long do I have to return a product and get my money back?"
print(f"问题①: {q1}\n检索到的上下文(喂给 LLM):")
for doc,s in retrieve(q1):
    tag="✓相关" if s>0.1 else "·无关"
    print(f"  [{tag} 相似度{s}] {doc}")
print("→ LLM 基于检索到的两条退款政策(30天 + 5-7个工作日)生成有依据的答案, 而非凭空编造\n")

# 情形②:知识库里没有答案 → 检索分数都很低 → 应识别并回答'我不知道'(而非幻觉)/ case ②: no answer → low scores → say "I don't know"
q2="What is the CEO's personal cell phone number?"
print(f"问题②(知识库里没有): {q2}\n检索结果:")
for doc,s in retrieve(q2): print(f"  [相似度{s}] {doc}")
top_sim=retrieve(q2)[0][1]
print(f"→ 最高相似度仅 {top_sim}(远低于阈值)! 好的 RAG 应识别'没有相关内容'→ 回答'知识库中无此信息'")
print("  而不是用不相关的块硬凑答案(那就是幻觉)。'检索不到就诚实说不知道'是防幻觉的关键。")


In [ ]:

# ============================================================
# 可视化:RAG 流水线 + 检索质量决定答案质量 / RAG pipeline + retrieval quality gates answer quality
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① RAG 流水线 / RAG pipeline
ax[0].axis("off"); ax[0].set_title("RAG 流水线",fontsize=12,weight="bold")
steps=[("离线: 文档切块 → embedding → 向量库+ANN 索引","#9467BD"),
       ("① 用户提问 → 编码成向量","#DD8452"),
       ("② 检索: 向量库找 top-k 相似块(混合语义+关键词)","#4C72B0"),
       ("③ 重排: cross-encoder 精排选最相关几块","#8172B3"),
       ("④ 生成: 问题+检索块 → LLM 生成答案(引用来源)","#55A868")]
for i,(s,c) in enumerate(steps):
    ax[0].add_patch(plt.Rectangle((0.03,0.8-i*0.16),0.94,0.13,fc=c,alpha=0.25,ec=c,transform=ax[0].transAxes))
    ax[0].text(0.5,0.865-i*0.16,s,ha="center",va="center",fontsize=8,transform=ax[0].transAxes)
    if i<4: ax[0].annotate("",xy=(0.5,0.8-i*0.16),xytext=(0.5,0.82-i*0.16),arrowprops=dict(arrowstyle="->"),transform=ax[0].transAxes)
# ② 检索质量决定答案质量 / retrieval quality gates answer quality
ax[1].axis("off"); ax[1].set_title("核心洞察: 检索质量决定答案质量",fontsize=12,weight="bold")
cases=[("检索到正确内容","→ LLM 有依据 → 正确答案 ✓","#55A868"),
       ("检索到错误/无关内容","→ LLM 基于错误资料 → 错误答案 ✗","#C44E52"),
       ("什么都没检索到","→ 应说'我不知道'; 若硬编 → 幻觉 ✗","#DD8452")]
for i,(a,b,c) in enumerate(cases):
    y=0.72-i*0.22
    ax[1].add_patch(plt.Rectangle((0.05,y-0.02),0.9,0.17,fc=c,alpha=0.2,ec=c,transform=ax[1].transAxes))
    ax[1].text(0.08,y+0.09,a,fontsize=9,weight="bold",color=c,transform=ax[1].transAxes)
    ax[1].text(0.08,y+0.02,b,fontsize=8.5,transform=ax[1].transAxes)
ax[1].text(0.5,0.05,"80% 的 RAG 工程功夫在'把检索做好', 而非选哪个 LLM",ha="center",fontsize=8.5,style="italic",transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/sd08_viz.png",dpi=80); plt.show()
print("左:RAG 流水线(切块→embedding→检索→重排→生成); 右:检索质量决定答案质量, 检索不到应诚实说不知道")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **RAG 最反直觉、也最重要的认知:难点在检索,不在生成**:几乎所有人第一次设计 RAG 都会把注意力放在"用 GPT-4 还是别的模型"上,但真正决定系统好坏的是**检索质量**。我们的 demo 一针见血:LLM 只能基于**检索到的内容**回答——检索到正确的退款政策,它就能给出正确答案;检索到无关内容,它就基于无关内容瞎答;什么都没检索到,它要么诚实说"不知道",要么(更糟)凭空编造一个听起来很自信的错误答案(幻觉)。**"垃圾进,垃圾出"在 RAG 里是字面意义的真理**。所以 RAG 系统 80% 的工程投入在检索侧:怎么切块、用什么 embedding、要不要混合语义 + 关键词检索(接 24.2)、要不要重排。面试里如果只谈 LLM 不谈检索,是抓错了重点。
2. **防幻觉的关键不是"更聪明的 LLM",而是"诚实的机制"**:RAG 常被宣传为"解决幻觉的方案",但**RAG 并不能完全消除幻觉**——即使给了正确的上下文,LLM 仍可能忽略它、或在上下文之外自由发挥。我们的 demo 展示了最危险的场景:知识库里**根本没有**某个问题的答案(比如"CEO 的私人电话")。此时检索分数全都很低,一个**设计良好**的 RAG 会识别"没有相关内容"并诚实回答"我不知道"或"知识库中无此信息";而一个**天真**的 RAG 会把这些不相关的块塞给 LLM,LLM 就可能用它们硬编一个答案——这正是幻觉。所以防幻觉靠三件事:①**检索分数阈值/相关性判断**(检索不到就别硬答);②**提示词工程**(明确要求 LLM"只基于提供的资料回答,没有依据就说不知道");③**引用来源**(让答案可溯源、可核验)。**"知道自己不知道"是 RAG 系统最重要也最难的能力。**
3. **诚实的深水区:RAG 的工程细节和评估都很难,而且没有银弹**。①**切块(chunking)是个又土又关键的问题**:块太大→检索不精准、浪费上下文;块太小→答案被切碎(我们 demo 里"30天"和"5-7个工作日"就在两个不同的块里,靠 top-k 才都召回)。切块策略(固定大小?按语义?按标题层级?重叠多少?)直接影响效果,却没有通用最优解,要针对文档类型调。②**评估极难**:RAG 有两层要分开评——**检索层**(相关内容有没有被召回?recall@k)和**生成层**(答案忠于检索内容吗?faithfulness;答案回答了问题吗?relevance)。生成质量还常常要用 LLM-as-judge 或 RAGAS 这类框架,本身就不完美。③**RAG vs 微调的选择**:要给模型**新知识**(公司文档、最新数据)用 RAG(动态、可更新、可溯源、便宜);要改模型的**行为/风格/格式**用微调;两者常结合。别把 RAG 能解决的问题(知识)用微调硬解(贵、静态、还是会幻觉)。④**工程现实**:知识库要增量更新(新文档进来要重新索引)、要做权限控制(用户只能检索到他有权看的文档,否则数据泄漏)、要处理多跳问题(答案需要综合多个文档)、表格/图片等非纯文本。⑤**前沿**:Agentic RAG(让 LLM 自主决定检索什么、多轮检索、自我批判)是当前热点。**结论:设计 RAG 系统的核心不是选哪个 LLM, 而是把检索做好——切块、embedding、混合检索、重排决定了答案质量的上限(检索到垃圾→垃圾答案), 80% 功夫在检索; 防幻觉靠检索阈值+提示词约束+引用来源, 让系统'检索不到就诚实说不知道'; 评估要分检索召回和生成忠实度两层; 加知识用 RAG、改行为用微调——这是当下最热的 LLM 系统设计题, 考的是对'检索是瓶颈'和'如何让 LLM 诚实'的深刻理解。**

**English**:
1. **RAG's most counterintuitive and important insight: the difficulty is in retrieval, not generation**: nearly everyone designing RAG for the first time focuses on "GPT-4 or another model," but what truly decides system quality is **retrieval quality**. Our demo cuts to it: the LLM can only answer based on **retrieved content** — retrieve the correct refund policy and it gives a correct answer; retrieve irrelevant content and it answers based on that; retrieve nothing and it either honestly says "I don't know" or (worse) fabricates a confident-sounding wrong answer (hallucination). **"Garbage in, garbage out" is literally true in RAG**. So 80% of RAG engineering goes to the retrieval side: how to chunk, which embedding, whether to hybridize semantic + keyword retrieval (per 24.2), whether to rerank. If you discuss only the LLM and not retrieval in an interview, you've missed the point.
2. **The key to preventing hallucination isn't "a smarter LLM" but "an honest mechanism"**: RAG is often marketed as "the solution to hallucination," but **RAG doesn't fully eliminate hallucination** — even given correct context, the LLM may ignore it or freewheel beyond it. Our demo shows the most dangerous scenario: the knowledge base **simply has no** answer to a question (e.g. "the CEO's personal phone"). Here retrieval scores are all low, and a **well-designed** RAG recognizes "no relevant content" and honestly answers "I don't know" or "no such info in the knowledge base"; a **naive** RAG stuffs those irrelevant chunks into the LLM, which may then fabricate an answer from them — exactly hallucination. So preventing hallucination relies on three things: ① **retrieval score thresholds / relevance judgment** (don't force an answer when nothing is retrieved); ② **prompt engineering** (explicitly instruct the LLM to "answer only from the provided material, say I don't know if there's no basis"); ③ **source citation** (make answers traceable and verifiable). **"Knowing what it doesn't know" is a RAG system's most important and hardest capability.**
3. **Honest deep end: RAG's engineering details and evaluation are hard, with no silver bullet**. ① **Chunking is an unglamorous but critical problem**: chunks too big → imprecise retrieval, wasted context; too small → answers get split (in our demo "30 days" and "5-7 business days" are in two different chunks, both recalled only via top-k). Chunking strategy (fixed size? by semantics? by heading hierarchy? how much overlap?) directly affects quality with no universal optimum, needing tuning per document type. ② **Evaluation is very hard**: RAG has two layers to evaluate separately — the **retrieval layer** (was relevant content recalled? recall@k) and the **generation layer** (is the answer faithful to retrieved content? faithfulness; does it answer the question? relevance). Generation quality often needs LLM-as-judge or RAGAS-like frameworks, themselves imperfect. ③ **RAG vs fine-tuning choice**: to give the model **new knowledge** (company docs, latest data) use RAG (dynamic, updatable, traceable, cheap); to change **behavior/style/format** use fine-tuning; often combined. Don't solve a knowledge problem (which RAG handles) with fine-tuning (expensive, static, still hallucinates). ④ **Engineering reality**: the knowledge base needs incremental updates (re-index new docs), permission control (users retrieve only docs they're authorized to see, else data leakage), multi-hop questions (answers synthesizing multiple docs), tables/images beyond plain text. ⑤ **Frontier**: Agentic RAG (the LLM decides what to retrieve, multi-round retrieval, self-critique) is a current hot topic. **Conclusion: designing a RAG system centers not on which LLM but on getting retrieval right — chunking, embedding, hybrid retrieval, reranking set the ceiling on answer quality (retrieve garbage → garbage answer), 80% of effort in retrieval; prevent hallucination with retrieval thresholds + prompt constraints + source citation so the system 'honestly says I don't know when nothing is retrieved'; evaluate retrieval recall and generation faithfulness as two layers; add knowledge with RAG, change behavior with fine-tuning — the hottest current LLM system-design question, testing deep understanding of 'retrieval is the bottleneck' and 'how to make the LLM honest.'**

> 💼 **实战视角 / Practical angle**
> **中文**:RAG 落地:①**切块**:按语义/标题层级切、合理大小(几百 token)+ 重叠, 针对文档类型调;②**检索**:好的 embedding 模型 + **混合检索**(语义 + BM25, 24.2)+ **重排**(cross-encoder);向量库用 pgvector/Pinecone/Weaviate/Qdrant/FAISS;③**防幻觉**:检索相关性阈值(检索不到就说不知道)、提示词约束"只基于资料回答"、**引用来源**;④**评估**:检索层 recall@k + 生成层忠实度/相关性(RAGAS / LLM-as-judge), 建评测集;⑤**工程**:增量索引更新、**权限过滤**(检索时按用户权限过滤)、多跳/查询改写、表格/图像处理;⑥**RAG vs 微调**:加知识 RAG、改行为微调, 常结合;⑦前沿: Agentic RAG(多轮/自主检索)、GraphRAG(知识图谱)。**答题**:先画流水线(切块→embed→检索→重排→生成), 强调检索决定质量+防幻觉机制+分层评估+权限。面试金句:*"RAG 在 LLM 回答前从知识库检索相关块塞进上下文, 解决私有/最新数据和幻觉; 流水线是切块→embedding→向量库检索(混合语义+关键词)→重排→生成引用来源; 核心是检索质量决定答案质量, 80%功夫在检索(切块/混合检索/重排); 防幻觉靠检索阈值+提示约束'没依据就说不知道'+引用来源; 评估分检索召回和生成忠实度; 加知识用RAG、改行为用微调, 还要权限过滤和增量更新。"*
> **English**: RAG in practice: ① **chunking**: by semantics/heading hierarchy, sensible size (a few hundred tokens) + overlap, tuned per document type; ② **retrieval**: a good embedding model + **hybrid retrieval** (semantic + BM25, 24.2) + **reranking** (cross-encoder); vector stores pgvector/Pinecone/Weaviate/Qdrant/FAISS; ③ **prevent hallucination**: retrieval-relevance thresholds (say I don't know if nothing retrieved), prompt constraint "answer only from the material," **cite sources**; ④ **evaluation**: retrieval-layer recall@k + generation-layer faithfulness/relevance (RAGAS / LLM-as-judge), build an eval set; ⑤ **engineering**: incremental index updates, **permission filtering** (filter by user access at retrieval), multi-hop/query rewriting, tables/images handling; ⑥ **RAG vs fine-tuning**: add knowledge with RAG, change behavior with fine-tuning, often combined; ⑦ frontier: Agentic RAG (multi-round/autonomous retrieval), GraphRAG (knowledge graphs). **Answering**: first draw the pipeline (chunk → embed → retrieve → rerank → generate), emphasize retrieval determines quality + anti-hallucination mechanisms + layered evaluation + permissions. Interview line: *"RAG retrieves relevant chunks from a knowledge base into the LLM's context before answering, solving private/latest data and hallucination; the pipeline is chunk → embedding → vector-DB retrieval (hybrid semantic + keyword) → rerank → generate with citations; the core is retrieval quality determines answer quality, 80% of effort in retrieval (chunking/hybrid/reranking); prevent hallucination with retrieval thresholds + prompt constraints 'say I don't know if no basis' + source citation; evaluate retrieval recall and generation faithfulness separately; add knowledge with RAG, change behavior with fine-tuning, plus permission filtering and incremental updates."*

---
### 小结 / Summary
- **中文**:RAG=回答前检索知识库相关块塞进 LLM 上下文, 解决私有/最新数据+减幻觉; 流水线: 切块→embedding→向量库检索→重排→生成引用。
- **English**: RAG = retrieve relevant chunks into the LLM's context before answering, solving private/latest data + reducing hallucination; pipeline: chunk → embedding → vector-DB retrieval → rerank → generate with citations.
- **中文**:核心洞察=检索质量决定答案质量(检索到垃圾→垃圾答案), 80%功夫在检索; 防幻觉靠检索阈值+提示约束+引用来源(检索不到诚实说不知道)。
- **English**: Core insight = retrieval quality determines answer quality (retrieve garbage → garbage answer), 80% of effort in retrieval; prevent hallucination with retrieval thresholds + prompt constraints + source citation (honestly say I don't know when nothing retrieved).
- **中文**:评估分检索召回和生成忠实度两层; 加知识用 RAG、改行为用微调; 还要权限过滤、增量更新、切块策略。
- **English**: Evaluate retrieval recall and generation faithfulness as two layers; add knowledge with RAG, change behavior with fine-tuning; plus permission filtering, incremental updates, chunking strategy.
